In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk

In [15]:
custo = pd.read_csv('../data/olist_customers_dataset.csv')
df_customers = custo.copy()
# df_customers.info()
# df_customers.isnull().sum()  
#99441 entries, Data columns (total 5 columns)

In [16]:
# # 중복 제거 
# # 중복 제거 전 행 수 저장
# before = len(df_customers)

# # customer_id 기준 중복 제거 #99441개 중복은 없음 
# df_customers = df_customers.drop_duplicates(subset=["customer_id"])

# # unique_id 기준 중복 체크 #3,345개
# df_customers['customer_unique_id'].nunique()

# # 1:다 관계 확인
# duplicates = df_customers.groupby('customer_unique_id')['customer_id'].nunique() # 2997개


In [17]:
#customer_unique_id 칼럼 삭제 _ 타 table과 조인할 때도 customer_id를 기준으로 진행될 것
#cusomer 개인 정보 보호를 위해서 더 찾아볼 수 있는 정보 없으므로 따로 사용할 가능성 낮음
df_customers = df_customers.drop(columns=['customer_unique_id'])
df_customers.info()
#(total 4 columns)

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_zip_code_prefix  99441 non-null  int64
 2   customer_city             99441 non-null  str  
 3   customer_state            99441 non-null  str  
dtypes: int64(1), str(3)
memory usage: 3.0 MB


In [18]:
# #데이터 정제
# # 대문자 혼재 확인 #[],Length: 0 #깔끔한 data, 정제 불필요 
# lowercase_state = df_customers[df_customers['customer_state'] != df_customers['customer_state'].str.upper()]
# print(lowercase_state['customer_state'].unique())

# # 소문자 혼재 확인 #[],Length: 0 #깔끔한 data, 정제 불필요 
# uppercase_city = df_customers[df_customers['customer_city'] != df_customers['customer_city'].str.lower()]
# print(uppercase_city['customer_city'].unique())

In [19]:
# # 주코드 대문자 통일
# df_customers["customer_state"] = df_customers["customer_state"].str.upper()

# # 도시명 소문자 통일
# df_customers["customer_city"] = df_customers["customer_city"].str.lower().str.strip()

In [20]:
# 데이터 정제 
# customer_city 포르투갈어 악센트 알파벳 정제

#이상치 여부 확인
#uv add unidecode
# from unidecode import unidecode

# # 정제 전후 비교
# df_customers['city_normalized'] = df_customers['customer_city'].apply(unidecode)

# # 다른 것들 찾기 #0 [] #깔끔한 data, 정제 불필요 
# accent_check = df_customers[df_customers['customer_city'] != df_customers['city_normalized']]
# print(len(accent_check))
# print(accent_check['customer_city'].unique())

In [21]:
# 'customer_state' 칼럼 state 길이가 2가 아닌 것 찾기
# 0 [], #깔끔한 data, 정제 불필요 
# invalid_state = df_customers[df_customers['customer_state'].str.len() != 2]
# print(len(invalid_state))
# print(invalid_state['customer_state'].unique())

In [22]:
# # state에 특수문자가 있는지 확인
# import re

# special_char_state = df_customers[df_customers['customer_state'].str.contains(r'[^A-Z0-9]', regex=True)]
# print(f"특수문자 있는 state: {len(special_char_state)}개")
# print(special_char_state['customer_state'].unique()) 

In [23]:
# # city에 특수문자가 있는지 확인 
# # (도시명 특성상, [공백, ', -]은 정상 특수문자로 허용해야함)
# # 공백, 하이픈(-), 어포스트로피(')를 제외하고 특수문자 찾기 #0개 처리 불필요
# def has_special_char(text):
#     import string
#     allowed = set(string.ascii_lowercase + string.digits + " -'")
#     return any(c not in allowed for c in str(text))

# special_char_city = df_customers[df_customers['customer_city'].apply(has_special_char)]
# print(len(special_char_city))
# print(special_char_city['customer_city'].unique()[:20])

In [24]:
# # zip_code_prefix 길이 확인 #분포 4,5만 존재 #정제 불필요
# print("zip_code_prefix 길이 분포:")
# print(df_customers['customer_zip_code_prefix'].astype(str).str.len().value_counts().sort_index())

In [ ]:
#도시명 정제 규칙 (모든 컬럼에 있는 도시명에 해당) 적용
#특수 문자 공백 1개와 ' 만 허용
def clean_city(series):
    return (
        series
        .str.lower()
        .str.strip()
        .str.replace(r"[^a-z\s']", " ", regex=True)  # a-z, 공백, '제외, 모든 문자를 공백
        .str.replace(r"\s+", " ", regex=True)        # 연속된 공백 > 공백 1개
    )

# 적용
df_customers['customer_city'] = clean_city(df_customers['customer_city'])